# β-VAE exploration

The loss is the per-image sum of squared error plus β times the closed-form KL. β = 1 is the standard VAE. The latent width is fixed at 8. Stage 07 fits β ∈ {0.1, 1, 4, 10} from the same initialization and writes `reports/mnist_betta_vae/beta_exploration.json`.

Read the three effects together:

- **Reconstruction.** Test MSE, posterior-mean decode. Smaller β can copy the pixels more closely. Larger β blurs them.
- **Disentanglement.** `active_units` counts latent axes whose posterior mean still varies across the test images (variance above 0.01). That is a compactness proxy. MNIST does not label stroke width or slant, so the traversal grids are the check: one row is one axis of the zero code, walked from −3 to 3. `traversal_response` is the mean pixel variance along that row.
- **Generation diversity.** Mean pairwise per-pixel squared distance among 64 prior samples. A collapsed generator scores lower.

`reports/mnist_betta_vae/plots/loss_vs_beta.png` is the β-weighted validation loss. A smaller β lowers that number by shrinking the KL term. It is not the disentanglement ranking. Run `make all` first if the report below is missing.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
for _candidate in (ROOT, *ROOT.parents):
    if (_candidate / "pyproject.toml").is_file():
        ROOT = _candidate
        break
report_path = ROOT / "reports" / "mnist_betta_vae" / "beta_exploration.json"
if not report_path.is_file():
    raise FileNotFoundError(f"missing {report_path}. Run make all from the repository root.")

report = json.loads(report_path.read_text(encoding="utf-8"))
frame = pd.DataFrame(report["runs"])[
    ["beta", "mse", "mae", "kl", "elbo", "active_units", "diversity"]
]
frame


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.2))
series = (
    ("mse", "reconstruction MSE"),
    ("active_units", "active latent units"),
    ("diversity", "generation diversity"),
)
for axis, (key, title) in zip(axes, series, strict=True):
    axis.plot(frame["beta"], frame[key], marker="o")
    axis.set_xlabel("beta")
    axis.set_title(title)
    axis.set_xticks(list(frame["beta"]))
fig.tight_layout()
plt.show()


In [ ]:
from IPython.display import Image, display

plots = ROOT / "reports" / "mnist_betta_vae" / "plots"
for row in report["runs"]:
    path = plots / row["traversal"]
    print(f"beta={row['beta']:g}  traversal_response={['%.4f' % value for value in row['traversal_response']]}")
    display(Image(filename=str(path)))


## This run

10 epochs, latent size 8, 1,000 training images and 200 test images of each class. Test numbers from `reports/mnist_betta_vae/beta_exploration.json`:

| β | MSE | KL | active units | diversity |
| --- | --- | --- | --- | --- |
| 0.1 | 0.091 | 26.94 | 8 | 0.168 |
| 1 | 0.094 | 15.03 | 8 | 0.226 |
| 4 | 0.111 | 8.37 | 7 | 0.204 |
| 10 | 0.148 | 3.94 | 6 | 0.151 |

Reconstruction MSE rises with β, and the β = 10 digits are blurrier. The per-axis KL at β = 0.1 sits in a narrow band, about 3.1 to 3.8, and every traversal row changes the digit. At β = 10 the KL collapses onto a few axes: z3, z4, and z5 are under 0.02 and their rows barely move, so `active_units` falls from 8 to 6. The rows that still move still change which digit you see, so this is unused axes turning off. MNIST does not label a continuous factor such as slant, and the grid is not a claim that one axis owns one factor.

Generation diversity, the mean pairwise per-pixel squared distance of 64 prior samples, peaks at β = 1 (0.226) and is lowest at β = 10 (0.151), where several latent directions no longer change the image. β = 0.1 is also tighter (0.168) even though its reconstructions are the sharpest. The Optuna study minimizes the β-weighted loss, so it selects β = 0.1. That is the best reconstruction in this sweep, not the most compact code.
